In [8]:
# Import libraries
import pandas as pd
import numpy as np
import os
import unicodedata
import re

In [9]:
# Change working directory
here = os.getcwd()
global_dir = os.path.dirname(here)
print(f"Working directory: {global_dir}")

# Set up paths
path = global_dir + "/1_extract_python"
path2 = global_dir + "/2_cleaning_stata"
path3 = global_dir + "/3_dataset"

# Create directories if they don't exist
os.makedirs(path2, exist_ok=True)
os.makedirs(path3, exist_ok=True)

Working directory: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Députés_français (Locuteurs)


In [10]:
# Cell 3: Load mandats dataset (SIMPLIFIED)
df_mandats = pd.read_csv(
    f"{path}/mandats.csv",
    low_memory=False  # Prevents DtypeWarning
)

print(f"Loaded mandats: {df_mandats.shape}")
print(f"Columns: {df_mandats.columns.tolist()}")
df_mandats.head()

Loaded mandats: (165634, 32)
Columns: ['Unnamed: 0', 'civ', 'prenom', 'nom', 'alpha', 'acteurRef', 'dateNais', 'villeNais', 'depNais', 'paysNais', 'profession_libelleCourant', 'profession_catSocPro', 'profession_famSocPro', 'mandat_type', 'mandat_uid', 'mandat_organe', 'mandat_type_organe', 'mandat_debut', 'mandat_fin', 'mandat_qualite', 'mandat_legislature', 'mandat_election_lieu_region', 'mandat_election_lieu_regionType', 'mandat_election_lieu_departement', 'mandat_election_lieu_numDepartement', 'mandat_election_lieu_numCirco', 'mandat_mandature_datePriseFonction', 'mandat_mandature_causeFin', 'mandat_mandature_premiereElection', 'mandat_mandature_placeHemicycle', 'mandat_mandature_mandatRemplaceRef', 'dateDeces']


,Unnamed: 0,civ,prenom,nom,alpha,acteurRef,dateNais,villeNais,depNais,paysNais,...,mandat_election_lieu_regionType,mandat_election_lieu_departement,mandat_election_lieu_numDepartement,mandat_election_lieu_numCirco,mandat_mandature_datePriseFonction,mandat_mandature_causeFin,mandat_mandature_premiereElection,mandat_mandature_placeHemicycle,mandat_mandature_mandatRemplaceRef,dateDeces
0,102901,M.,Didier,Le Gac,Le Gac,PA719404,1965-07-19,Castres,Tarn,France,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,104360,Mme,Liliana,Tanguy,Tanguy,PA719550,1967-03-12,Pancevo,NaN,Serbie,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,115719,M.,Bastien,Lachaud,Lachaud,PA720846,1980-08-05,Vitry sur Seine,Val-de-Marne,France,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,116123,Mme,Mathilde,Panot,Panot,PA720892,1989-01-15,Tours,Indre-et-Loire,France,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,154365,M.,Pierre,Cazeneuve,Cazeneuvep,PA795864,1995-03-11,Saint Cloud,Hauts-de-Seine,France,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
df.columns

Index(['Unnamed: 0', 'civ', 'prenom', 'nom', 'alpha', 'acteurRef', 'dateNais',
       'villeNais', 'depNais', 'paysNais', 'profession_libelleCourant',
       'profession_catSocPro', 'profession_famSocPro', 'mandat_type',
       'mandat_id', 'organe_id', 'mandat_type_organe', 'mandat_debut',
       'mandat_fin', 'mandat_qualite', 'mandat_legislature',
       'mandat_election_lieu_region', 'mandat_election_lieu_regionType',
       'mandat_election_lieu_departement',
       'mandat_election_lieu_numDepartement', 'mandat_election_lieu_numCirco',
       'mandat_mandature_datePriseFonction', 'mandat_mandature_causeFin',
       'mandat_mandature_premiereElection', 'mandat_mandature_placeHemicycle',
       'mandat_mandature_mandatRemplaceRef', 'dateDeces'],
      dtype='object')

In [13]:
# Cell 4: DEBUG - Check actual column names
df = df_mandats.copy()

print("=== ACTUAL COLUMN NAMES ===")
print(df.columns.tolist())

# Check if columns exist
print("\n=== CHECKING FOR KEY COLUMNS ===")
for col in ['acteurref', 'mandat_uid', 'mandat_organe']:
    if col in df.columns:
        print(f"✓ Found: {col}")
    else:
        print(f"✗ Missing: {col}")
        # Find similar names
        similar = [c for c in df.columns if col.lower() in c.lower()]
        if similar:
            print(f"  Possible alternatives: {similar}")

=== ACTUAL COLUMN NAMES ===
['Unnamed: 0', 'civ', 'prenom', 'nom', 'alpha', 'acteurRef', 'dateNais', 'villeNais', 'depNais', 'paysNais', 'profession_libelleCourant', 'profession_catSocPro', 'profession_famSocPro', 'mandat_type', 'mandat_uid', 'mandat_organe', 'mandat_type_organe', 'mandat_debut', 'mandat_fin', 'mandat_qualite', 'mandat_legislature', 'mandat_election_lieu_region', 'mandat_election_lieu_regionType', 'mandat_election_lieu_departement', 'mandat_election_lieu_numDepartement', 'mandat_election_lieu_numCirco', 'mandat_mandature_datePriseFonction', 'mandat_mandature_causeFin', 'mandat_mandature_premiereElection', 'mandat_mandature_placeHemicycle', 'mandat_mandature_mandatRemplaceRef', 'dateDeces']

=== CHECKING FOR KEY COLUMNS ===
✗ Missing: acteurref
  Possible alternatives: ['acteurRef']
✓ Found: mandat_uid
✓ Found: mandat_organe


In [14]:
# Cell 4: Clean and rename ID variables
df = df_mandats.copy()

# Drop unnecessary columns
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

# Rename ID columns avec les VRAIS noms
rename_dict = {
    'acteurRef': 'acteur_id',          # ← Capital R !
    'mandat_uid': 'mandat_id',
    'mandat_organe': 'organe_id'
}
df = df.rename(columns=rename_dict)

print("IDs renamed successfully")
print(df[['acteur_id', 'mandat_id', 'organe_id']].head())

IDs renamed successfully
  acteur_id mandat_id organe_id
0  PA719404  PM876261   PO59051
1  PA719550  PM876330  PO872098
2  PA720846  PM876243  PO872098
3  PA720892  PM876241  PO872098
4  PA795864  PM876329  PO872098


In [15]:
# Cell 5: Clean and rename identity variables
rename_ident = {
    'civ': 'ident_civ',
    'prenom': 'ident_prenom',
    'nom': 'ident_nom',
    'alpha': 'ident_alpha',
    'dateNais': 'ident_datenais',        
    'dateDeces': 'ident_datedeces',      
    'villeNais': 'ident_villenais',      
    'depNais': 'ident_depnais',          
    'paysNais': 'ident_paysnais'         
}
df = df.rename(columns=rename_ident)

print("Identity variables renamed")
print(df[[col for col in rename_ident.values() if col in df.columns]].head())

Identity variables renamed
  ident_civ ident_prenom  ident_nom ident_alpha ident_datenais  \
0        M.       Didier     Le Gac      Le Gac     1965-07-19   
1       Mme      Liliana     Tanguy      Tanguy     1967-03-12   
2        M.      Bastien    Lachaud     Lachaud     1980-08-05   
3       Mme     Mathilde      Panot       Panot     1989-01-15   
4        M.       Pierre  Cazeneuve  Cazeneuvep     1995-03-11   

  ident_datedeces  ident_villenais   ident_depnais ident_paysnais  
0             NaN          Castres            Tarn         France  
1             NaN          Pancevo             NaN         Serbie  
2             NaN  Vitry sur Seine    Val-de-Marne         France  
3             NaN            Tours  Indre-et-Loire         France  
4             NaN      Saint Cloud  Hauts-de-Seine         France  


In [21]:
# Cell 6: Clean and rename profession variables
rename_profession = {
    'profession_libelleCourant': 'profession_libellecourant',
    'profession_catSocPro': 'profession_catsocpro',
    'profession_famSocPro': 'profession_famsocpro'
}
df = df.rename(columns=rename_profession)

print("Profession variables renamed")
print(df[['profession_libellecourant', 'profession_catsocpro', 'profession_famsocpro']].head())

Profession variables renamed
               profession_libellecourant  \
0  Profession rattachée à l'enseignement   
1      Cadre supérieur du secteur public   
2  Professeur du secondaire et technique   
3   Coordinatrice de projets associatifs   
4          Cadre de la fonction publique   

                                profession_catsocpro  \
0  Professions de l’enseignement primaire et prof...   
1  Cadres administratifs et techniques de la fonc...   
2  Cadres administratifs et techniques de la fonc...   
3  Cadres des services administratifs et commerci...   
4  Cadres administratifs et techniques de la fonc...   

                                profession_famsocpro  
0                         Professions intermédiaires  
1  Cadres et professions intellectuelles supérieures  
2  Cadres et professions intellectuelles supérieures  
3  Cadres et professions intellectuelles supérieures  
4  Cadres et professions intellectuelles supérieures  


In [22]:
# Cell 7: Clean and rename mandate variables
# Drop unnecessary columns (use REAL column names from CSV)
cols_to_drop = [
    'mandat_mandature_premiereElection',      # ← Real names !
    'mandat_mandature_mandatRemplaceRef',
    'mandat_type_organe',
    'mandat_election_lieu_regionType'         # ← Add this too
]
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Rename mandate variables with correct camelCase
rename_mandat = {
    'mandat_election_lieu_region': 'election_lieu_region',
    'mandat_election_lieu_departement': 'election_lieu_departement',
    'mandat_election_lieu_numDepartement': 'election_lieu_numdepartem',      # ← Real name !
    'mandat_election_lieu_numCirco': 'election_lieu_numcirco',                # ← Real name !
    'mandat_mandature_datePriseFonction': 'mandature_dateprisefoncti',        # ← Real name !
    'mandat_mandature_causeFin': 'mandature_causefin',                        # ← Real name !
    'mandat_mandature_placeHemicycle': 'mandature_placehemicycle'             # ← Real name !
}
df = df.rename(columns=rename_mandat)

print("Mandate variables cleaned and renamed")

Mandate variables cleaned and renamed


In [24]:
# Cell 8: Create INSEE profession codes - PCS 2003 (8 groups)
def map_pcs_2003(famsocpro):
    """Map profession family to PCS 2003 code (8 groups)"""
    mapping = {
        "Agriculteurs exploitants": 1,
        "Artisans, commerçants et chefs d'entreprise": 2,
        "Cadres et professions intellectuelles supérieures": 3,
        "Professions Intermédiaires": 4,
        "Employés": 5,
        "Ouvriers": 6,
        "Retraités": 7,
        "Autres personnes sans activité professionnelle": 8
    }
    return mapping.get(famsocpro, np.nan)

df['profession_PCS_2003'] = df['profession_famsocpro'].apply(map_pcs_2003)
print("PCS 2003 (8 groups) created")
print(df['profession_PCS_2003'].value_counts().sort_index())


PCS 2003 (8 groups) created
profession_PCS_2003
1.0      3091
2.0      6487
3.0    107130
4.0      6202
5.0      4284
6.0      1010
7.0      6944
8.0       157
Name: count, dtype: int64


In [25]:
# Cell 9: Create INSEE profession codes - PCS 2003 (42 groups, 3rd level)
def map_pcs3_2003(catsocpro):
    """Map profession category to PCS 2003 3-digit code (42 groups)"""
    mapping = {
        "Agriculteurs exploitants": 10,
        "Artisans": 21,
        "Commerçants et assimilés": 22,
        "Chefs d'entreprise de 10 salariés ou plus": 23,
        "Professions libérales et assimilés": 31,
        "Cadres de la fonction publique, professions intellectuelles et artistiques": 32,
        "Cadres d'entreprise": 36,
        "Professions intermédiaires de l'enseignement, de la santé, de la fonction publique et assimilés": 41,
        "Professions intermédiaires administratives et commerciales des entreprises": 46,
        "Techniciens": 47,
        "Contremaîtres, agents de maîtrise": 48,
        "Employés de la fonction publique": 51,
        "Employés administratifs d'entreprise": 54,
        "Employés de commerce": 55,
        "Personnels des services directs aux particuliers": 56,
        "Ouvriers qualifiés": 61,
        "Ouvriers non qualifiés": 66,
        "Ouvriers agricoles": 69,
        "Anciens agriculteurs exploitants": 71,
        "Anciens artisans, commerçants, chefs d'entreprise": 72,
        "Anciens cadres et professions intermédiaires": 73,
        "Anciens employés et ouvriers": 76,
        "Chômeurs n'ayant jamais travaillé": 81,
        "Inactifs divers (autres que retraités)": 82
    }
    return mapping.get(catsocpro, np.nan)

df['profession_PCS3_2003'] = df['profession_catsocpro'].apply(map_pcs3_2003)
print("PCS 2003 (42 groups) created")
print(df['profession_PCS3_2003'].value_counts().sort_index())

PCS 2003 (42 groups) created
profession_PCS3_2003
10.0     3091
21.0      584
22.0     3068
23.0     4559
31.0    18720
36.0    21212
41.0     5199
46.0     1063
47.0      765
48.0       12
51.0      541
54.0     1945
55.0      726
56.0      216
61.0      236
69.0      395
71.0      138
72.0      302
73.0     6464
76.0       40
82.0      157
Name: count, dtype: int64


In [26]:
# Cell 10: Create INSEE profession codes - PCS 2020 (6 groups)
def map_pcs_2020(famsocpro):
    """Map profession family to PCS 2020 code (6 groups)"""
    mapping = {
        "Agriculteurs exploitants": 1,
        "Artisans, commerçants et chefs d'entreprise": 2,
        "Cadres et professions intellectuelles supérieures": 3,
        "Professions Intermédiaires": 4,
        "Employés": 5,
        "Ouvriers": 6
    }
    return mapping.get(famsocpro, np.nan)

df['profession_PCS_2020'] = df['profession_famsocpro'].apply(map_pcs_2020)
print("PCS 2020 (6 groups) created")
print(df['profession_PCS_2020'].value_counts().sort_index())


PCS 2020 (6 groups) created
profession_PCS_2020
1.0      3091
2.0      6487
3.0    107130
4.0      6202
5.0      4284
6.0      1010
Name: count, dtype: int64


In [27]:
# Cell 11: Create gender variable
df['ident_male'] = df['ident_civ'].map({"M.": 1, "Mme": 0})
print("Gender variable created")
print(df['ident_male'].value_counts())

Gender variable created
ident_male
1    117055
0     48579
Name: count, dtype: int64


In [28]:
# Cell 12: Create birth year and age variables
df['ident_year_birth'] = pd.to_datetime(df['ident_datenais'], errors='coerce').dt.year
df['ident_age'] = 2023 - df['ident_year_birth']
print("Birth year and age variables created")
print(df[['ident_datenais', 'ident_year_birth', 'ident_age']].head(10))

# Cell 13: Clean birth country variable
def normalize_string(s):
    """Normalize French accents"""
    if pd.isna(s):
        return s
    s = str(s).strip()
    nfd = unicodedata.normalize('NFD', s)
    return ''.join(c for c in nfd if unicodedata.category(c) != 'Mn').title()

df['ident_paysnais'] = df['ident_paysnais'].apply(normalize_string)

# Replace specific values
df.loc[df['ident_paysnais'] == 'Fr', 'ident_paysnais'] = 'France'
df.loc[df['ident_paysnais'] == 'Polynesie Francaise', 'ident_paysnais'] = 'France'
df.loc[~df['ident_depnais'].isna(), 'ident_paysnais'] = 'France'

print("Birth country cleaned")
print(df['ident_paysnais'].value_counts())

# Cell 14: Create abroad birth indicator
df['ident_born_abroad'] = (df['ident_paysnais'] != 'France').astype(int)
print("Abroad birth indicator created")
print(df['ident_born_abroad'].value_counts())

Birth year and age variables created
  ident_datenais  ident_year_birth  ident_age
0     1965-07-19            1965.0       58.0
1     1967-03-12            1967.0       56.0
2     1980-08-05            1980.0       43.0
3     1989-01-15            1989.0       34.0
4     1995-03-11            1995.0       28.0
5     1988-05-26            1988.0       35.0
6     1992-05-14            1992.0       31.0
7     1972-01-23            1972.0       51.0
8     1966-05-29            1966.0       57.0
9     1965-03-12            1965.0       58.0
Birth country cleaned
ident_paysnais
France                   140261
Algerie                    1994
Maroc                      1065
Tunisie                     904
Senegal                     887
Allemagne                   489
Espagne                     466
Grece                       398
Congo                       255
Belgique                    249
Serbie                      216
Coree Du Sud                206
Burkina Faso                206
Cana

In [29]:
# Cell 15: Encode department number
df['election_lieu_numdepartem'] = pd.Categorical(df['election_lieu_numdepartem']).codes
print("Department number encoded")

# Cell 16: Create mandate year variables
df['mandat_annee_debut'] = pd.to_datetime(df['mandat_debut'], errors='coerce').dt.year
df['mandat_annee_fin'] = pd.to_datetime(df['mandat_fin'], errors='coerce').dt.year
print("Mandate year variables created")
print(df[['mandat_debut', 'mandat_fin', 'mandat_annee_debut', 'mandat_annee_fin']].head())

# Cell 17: Rename variables with source suffix
vars_to_rename = [
    'ident_civ', 'ident_prenom', 'ident_nom', 'ident_alpha', 'ident_datenais',
    'ident_villenais', 'ident_depnais', 'ident_paysnais', 'ident_datedeces',
    'ident_male', 'ident_year_birth', 'ident_age', 'ident_born_abroad',
    'profession_libellecourant', 'profession_catsocpro', 'profession_famsocpro',
    'mandat_type', 'mandat_debut', 'mandat_fin', 'mandat_qualite', 'mandat_legislature',
    'mandat_annee_debut', 'mandat_annee_fin', 'mandature_dateprisefoncti',
    'mandature_causefin', 'mandature_placehemicycle', 'election_lieu_region',
    'election_lieu_departement', 'election_lieu_numcirco', 'election_lieu_numdepartem'
]
rename_suffix = {var: f"{var}_ANM" for var in vars_to_rename if var in df.columns}
df = df.rename(columns=rename_suffix)

print("Source suffix added (ANM = Assemblée Nationale Mandats)")
print(df.columns.tolist())

Department number encoded
Mandate year variables created
  mandat_debut mandat_fin  mandat_annee_debut  mandat_annee_fin
0   2025-12-06        NaN                2025               NaN
1   2025-12-06        NaN                2025               NaN
2   2025-12-06        NaN                2025               NaN
3   2025-12-06        NaN                2025               NaN
4   2025-12-06        NaN                2025               NaN
Source suffix added (ANM = Assemblée Nationale Mandats)
['ident_civ_ANM', 'ident_prenom_ANM', 'ident_nom_ANM', 'ident_alpha_ANM', 'acteur_id', 'ident_datenais_ANM', 'ident_villenais_ANM', 'ident_depnais_ANM', 'ident_paysnais_ANM', 'profession_libellecourant_ANM', 'profession_catsocpro_ANM', 'profession_famsocpro_ANM', 'mandat_type_ANM', 'mandat_id', 'organe_id', 'mandat_debut_ANM', 'mandat_fin_ANM', 'mandat_qualite_ANM', 'mandat_legislature_ANM', 'election_lieu_region_ANM', 'election_lieu_departement_ANM', 'election_lieu_numdepartem_ANM', 'election_lieu

In [ ]:
# Cell 18: Save mandats dataset (CSV only)
df.to_csv(f"{path2}/mandats.csv", index=False)
print(f"Mandats saved to {path2}/mandats.csv")

# Cell 19: Load organes dataset
df_organes = pd.read_csv(f"{path}/organes.csv")
print(f"Loaded organes: {df_organes.shape}")

# Cell 20: Clean and rename organes variables
if 'Unnamed: 0' in df_organes.columns:
    df_organes = df_organes.drop('Unnamed: 0', axis=1)

rename_organes = {
    'codetype': 'organe_codetype',
    'uid': 'organe_id',
    'libelleabrege': 'organe_abrege',
    'libelle': 'organe_libelle'
}
df_organes = df_organes.rename(columns=rename_organes)

print("Organes variables renamed")

# Add source suffix
organes_vars = ['organe_codetype', 'organe_abrege', 'organe_libelle']
rename_organes_suffix = {var: f"{var}_ANO" for var in organes_vars if var in df_organes.columns}
df_organes = df_organes.rename(columns=rename_organes_suffix)

# Save cleaned organes
df_organes.to_csv(f"{path2}/organes.csv", index=False)
print("Organes cleaned and saved")
print(df_organes.columns.tolist())

# Cell 21: Merge mandats with organes
df_merged = pd.merge(
    df_organes,
    df,
    on='organe_id',
    how='inner'
)

print(f"Merged dataset shape: {df_merged.shape}")
print(f"Number of deputies: {df_merged['acteur_id'].nunique()}")

# Cell 22: Reorder columns
key_cols = [col for col in df_merged.columns if col.startswith(('mandat_id', 'acteur_id', 'organe_id', 'ident_', 'profession_', 'mandat_', 'mandature_', 'election_', 'organe_'))]
other_cols = [col for col in df_merged.columns if col not in key_cols]

df_merged = df_merged[key_cols + other_cols]

print("Columns reordered")

Mandats saved to c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Députés_français (Locuteurs)/2_cleaning_stata/mandats.csv
Loaded organes: (10752, 5)
Organes cleaned and saved
['organe_id', 'codeType', 'libelleAbrege', 'organe_libelle']
Organes cleaned and saved
['organe_id', 'codeType', 'libelleAbrege', 'organe_libelle_ANO']
Merged dataset shape: (165465, 39)
Number of deputies: 3092
  organe_id    codeType        libelleAbrege  \
0  PO191887  ORGEXTPARL  Mines antipersonnel   
1  PO191887  ORGEXTPARL  Mines antipersonnel   
2  PO191887  ORGEXTPARL  Mines antipersonnel   
3  PO191887  ORGEXTPARL  Mines antipersonnel   
4  PO191887  ORGEXTPARL  Mines antipersonnel   

                                  organe_libelle_ANO ident_civ_ANM  \
0  Commission nationale pour l'élimination des mi...            M.   
1  Commission nationale pour l'élimination des mi...           Mme   
2  Commission nationale pour l'élimination des mi...   

In [33]:
# Cell 23: Save final dataset (CSV only)
df_merged.to_csv(f"{path3}/mandats.csv", index=False)
print(f"Final dataset saved to: {path3}/mandats.csv")
print(f"Final dataset shape: {df_merged.shape}")
print(f"Total columns: {len(df_merged.columns)}")

# Cell 24: Summary statistics
print("\n=== SUMMARY STATISTICS ===")
print(f"Gender distribution:\n{df_merged['ident_male_ANM'].value_counts()}")
print(f"\nBirth country distribution:\n{df_merged['ident_paysnais_ANM'].value_counts().head(10)}")
print(f"\nPCS 2003 distribution:\n{df_merged['profession_PCS_2003'].value_counts().sort_index()}")
print(f"\nAge statistics:\n{df_merged['ident_age_ANM'].describe()}")

Final dataset saved to: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Députés_français (Locuteurs)/3_dataset/mandats.csv
Final dataset shape: (165465, 39)
Total columns: 39

=== SUMMARY STATISTICS ===
Gender distribution:
ident_male_ANM
1    116938
0     48527
Name: count, dtype: int64

Birth country distribution:
ident_paysnais_ANM
France       140103
Algerie        1991
Maroc          1065
Tunisie         904
Senegal         887
Allemagne       489
Espagne         466
Grece           398
Congo           255
Belgique        247
Name: count, dtype: int64

PCS 2003 distribution:
profession_PCS_2003
1.0      3087
2.0      6475
3.0    107011
4.0      6194
5.0      4282
6.0      1010
7.0      6934
8.0       156
Name: count, dtype: int64

Age statistics:
count    165384.000000
mean         60.907627
std          14.299153
min          21.000000
25%          51.000000
50%          62.000000
75%          72.000000
max         106.000